# Task 5 — Capstone: redesign one chart for the human visual system

**Question:** How does diamond price change as carat increases?

The provided `diamonds.csv` has 53,940 rows and 10 columns. This notebook uses every row to calculate median prices in 0.1-carat bins. A reproducible random sample of 2,500 rows is shown only as grey context in the redesign. No rows are excluded from the summary; the supplied data has no missing values.

## 1. Start from a bad chart
The starting chart is a red, overplotted scatter of 5,000 sampled diamonds. Its purpose is to make the problem visible: saturation and overlap compete with the relationship we want to understand.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook-friendly path: keep data/diamonds.csv next to this notebook in a data folder.
HERE = Path.cwd()
DATA = HERE / "data" / "diamonds.csv"
if not DATA.exists():
    # Also support running the notebook from the packaged project directory.
    DATA = HERE / "week2_task5_perception_redesign" / "data" / "diamonds.csv"
df = pd.read_csv(DATA)
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
print("Missing values:", int(df.isna().sum().sum()))
display(df.head())


## 2. Perception audit — before redesign

| Row | Audit of BAD EXAMPLE |
|---|---|
| Preattentive | Position (carat and price), colour hue, and point density are immediately visible. Red hue does not encode a meaningful variable; it adds noise rather than helping answer the question. |
| Pop-out | The saturated red cloud attracts attention everywhere, so no single finding pops out. The title only says “BAD EXAMPLE” and does not state a result. |
| Gestalt | **Proximity** groups nearby points into a dense cloud, but overplotting makes local density and individual values difficult to separate. |
| Cognitive load | **Intrinsic:** interpreting the carat–price relationship and the spread. **Extraneous:** 5,000 overlapping red marks, no summary, weak hierarchy. **Germane:** comparing x and y positions to infer association. |
| Channels | Carat → horizontal position (high-ranking quantitative channel). Price → vertical position (high-ranking quantitative channel). Red hue → colour (not useful for these numeric variables). Point density is an unintended secondary cue. |
| Working memory | More than ~4 chunks: the reader must mentally track the cloud, possible clusters, outliers, spread, and overall trend at once. |


In [ ]:
rng = np.random.default_rng(42)
bad_n = min(5000, len(df))
bad = df.iloc[rng.choice(len(df), size=bad_n, replace=False)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(bad["carat"], bad["price"], s=16, alpha=0.65, color="#d62728", edgecolors="none")
ax.set_title("BAD EXAMPLE", fontsize=16, weight="bold")
ax.set_xlabel("Carat")
ax.set_ylabel("Price (USD)")
ax.text(0.01, -0.18,
        f"Question: How does diamond price change as carat increases? Displayed sample: {bad_n:,} of {len(df):,} rows (seed=42).",
        transform=ax.transAxes, fontsize=9, va="top")
fig.tight_layout()
fig.savefig(HERE / "bad_example.png", dpi=300, bbox_inches="tight")
plt.show()


## 3. Rebuild it: one figure, one message

**Finding-led title:** Median diamond price rises with carat, with a steeper climb above 1 ct.

- **One intended pop-out:** the blue median line. Context points are grey.
- **Highest-ranking channel:** quantitative position on common x and y axes.
- **Named Gestalt principle:** **similarity** groups grey context points together; the continuous blue line is perceived as a separate summary series.
- **No colour-only encoding:** the summary is also identified by line continuity, circular markers, and a legend label.
- **Chunking:** three chunks — grey distribution, blue median trend, one takeaway.
- **Trade-off:** greying and sampling the background makes individual prices, local clusters, and outliers in the full dataset harder to inspect. The trend is easier to see, but distribution detail is sacrificed.


In [ ]:
context_n = min(2500, len(df))
context = df.iloc[rng.choice(len(df), size=context_n, replace=False)].copy()
summary = df.assign(carat_bin=(np.floor(df["carat"] * 10) / 10).round(1))
med = summary.groupby("carat_bin", observed=True).agg(
    median_price=("price", "median"), n=("price", "size")
).reset_index()
med = med[(med["carat_bin"] >= 0.1) & (med["carat_bin"] <= 3.0)]
med.to_csv(HERE / "median_price_by_carat_bin.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(context["carat"], context["price"], s=10, color="#b9bec5", alpha=0.35,
           edgecolors="none", rasterized=True, label="Sampled individual diamonds")
ax.plot(med["carat_bin"], med["median_price"], color="#1769aa", linewidth=2.8,
        marker="o", markersize=4, label="Median price per 0.1-carat bin")
ax.set_title("Median diamond price rises with carat, with a steeper climb above 1 ct",
             fontsize=15, weight="bold", loc="left", pad=12)
ax.set_xlabel("Carat (0.1-carat bins; lower edge used for median)")
ax.set_ylabel("Price (USD; medians use all rows in each bin)")
ax.grid(axis="y", color="#e5e7eb", linewidth=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper left")
ax.text(0.01, -0.22,
        f"n = {len(df):,} rows analyzed; no missing values. Grey context: {context_n:,} random rows (seed=42); blue medians use all rows. Bins: 0.1 carat.",
        transform=ax.transAxes, fontsize=8.5, va="top", color="#555555")
fig.tight_layout()
fig.savefig(HERE / "perception_redesign.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved 300 DPI charts and median_price_by_carat_bin.csv.")


## 4. Perception audit — after redesign

| Row | Audit of redesign |
|---|---|
| Preattentive | Position carries both quantitative variables. Grey points establish the distribution; one blue median line is the single strong-colour focal element. |
| Pop-out | The blue median line is the only intended pop-out and directly supports the finding-led title. It is explicitly labelled in the legend. |
| Gestalt | **Similarity** groups grey points as context, while the continuous blue line is perceived as one summary series. |
| Cognitive load | **Intrinsic:** carat–price association and variation. **Extraneous:** removed red saturation and most overplotting; light grid is restrained. **Germane:** the median line compresses the trend into a readable summary. |
| Channels | Carat → horizontal position; price → vertical position; both are high-ranking quantitative channels. Blue hue separates summary from grey context but is redundant with line continuity, markers, and legend label. |
| Working memory | About 3 chunks: grey distribution, blue median trend, and the finding (steeper increase above 1 ct). |


## 5. Test on a real human reader — do this after the chart appears

Show **only `perception_redesign.png`** to someone who has not seen the data. Ask: “In one sentence, what does this chart show?” Record their sentence verbatim below. Do not invent or polish their words.

- **Reader's verbatim sentence:** `[FILL IN AFTER A REAL TEST]`
- **Does it match the title?** `[YES / PARTLY / NO — explain]`
- **What misled them, if anything?** `[FILL IN]`
- **What would you change next?** `[FILL IN]`

A real reader response cannot be generated from the dataset; this section must be completed after the human test.
